<a href="https://colab.research.google.com/github/nmwiley808/csci198-Music-Intelligence-with-Deep-Learning-Senior-Project/blob/main/notebooks/03_CNN_GTZAN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CNN on GTZAN

## Description

This notebook trains a baseline Convolutional Neural Network (CNN) on GTZAN log-mel spectrogram features.

Pipeline:
- Precompute and cache log-mel features (if not already cached)
- Load feature tensors
- Train CNN classifier
- Evaluate baseline accuracy

This model establishes the CNN baseline for later comparison with LSTM and Transformer architectures.

In [5]:
# Setup
from google.colab import drive
drive.mount('/content/drive')

import os
import librosa
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

PROJECT_PATH = "/content/drive/MyDrive/csci198/csci198-Music-Intelligence-with-Deep-Learning-Senior-Project"
os.chdir(PROJECT_PATH)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda


In [3]:
# Parameters
TARGET_SR = 22050
DURATION = 30
SAMPLES_PER_TRACK = TARGET_SR * DURATION
N_MELS = 128
BATCH_SIZE = 32
EPOCHS = 10

In [4]:
# Feature Cache Paths
X_PATH = "data/processed/X_gtzan.npy"
Y_PATH = "data/processed/Y_gtzan.npy"

os.makedirs("data/processed", exist_ok=True)

In [10]:
# Build Cache If Missing
if not os.path.exists(X_PATH):
  print("Building GTZAN feature cache...")
  def load_audio(file_path):
    y, sr = librosa.load(file_path, sr=TARGET_SR)
    if len(y) > SAMPLES_PER_TRACK:
      y = y[:SAMPLES_PER_TRACK]
    else:
      y = np.pad(y, (0, SAMPLES_PER_TRACK - len(y)))
    return y

  def extract_log_mel(y):
    mel = librosa.feature.melspectrogram(y=y, sr=TARGET_SR, n_mels=N_MELS)
    return librosa.power_to_db(mel, ref=np.max)

  gtzan_root = "data/raw/gtzan"

  X = []
  labels = []

  for root, dirs, files in os.walk(gtzan_root):
    for file in tqdm(files):
      if file.endswith(".wav"):
        label = os.path.basename(root)
        file_path = os.path.join(root, file)

        try:
          y_audio = load_audio(file_path)
          log_mel = extract_log_mel(y_audio)

          X.append(log_mel)
          labels.append(label)
        except Exception as e:
          print(f"Skipping file {file_path} due to error: {e}")
          continue

  X = np.array(X)
  le = LabelEncoder()
  Y = le.fit_transform(labels)

  np.save(X_PATH, X)
  np.save(Y_PATH, Y)

  print("Cache saved.")
else:
  print("Cache already exists.")

Building GTZAN feature cache...


100%|██████████| 2/2 [00:00<00:00, 39945.75it/s]
0it [00:00, ?it/s]
 48%|████▊     | 48/100 [00:02<00:02, 19.72it/s]/tmp/ipython-input-965361272.py:5: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(file_path, sr=TARGET_SR)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
 53%|█████▎    | 53/100 [00:02<00:03, 15.05it/s]

Skipping file data/raw/gtzan/Data/genres_original/jazz/jazz.00054.wav due to error: 


100%|██████████| 100/100 [00:11<00:00,  8.46it/s]
0it [00:00, ?it/s]
100%|██████████| 100/100 [00:00<00:00, 961996.33it/s]


Cache saved.


In [11]:
# Load Cached Data
X = np.load(X_PATH)
Y = np.load(Y_PATH)

print("X shape:", X.shape)
print("Y shape:", Y.shape)

X shape: (999, 128, 1292)
Y shape: (999,)


In [13]:
# Train & Test Split
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

# Add channel dimension
X_train = torch.tensor(X_train).unsqueeze(1).float()
X_test = torch.tensor(X_test).unsqueeze(1).float()
Y_train = torch.tensor(Y_train)
Y_test = torch.tensor(Y_test)

print ("Train shape:", X_train.shape)

Train shape: torch.Size([799, 1, 128, 1292])


In [16]:
# Define CNN Model
class CNN(nn.Module):
  def __init__(self, num_classes):
    super(CNN, self).__init__()

    self.conv = nn.Sequential(
        nn.Conv2d(1, 16, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),

        nn.Conv2d(16, 32, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),

    )

    self.fc = nn.Sequential(
        nn.Flatten(),
        nn.Linear(32 * (N_MELS//4) * (X.shape[2]//4), 128),
        nn.ReLU(),
        nn.Linear(128, num_classes)

    )

  def forward(self, x):
    x = self.conv(x)
    x = self.fc(x)
    return x

In [18]:
# Train
model = CNN(num_classes=len(np.unique(Y))).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Create a DataLoader for batch processing
train_dataset = torch.utils.data.TensorDataset(X_train, Y_train)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

# Move model to device
model.to(device)

for epoch in range(EPOCHS):
  model.train()
  running_loss = 0.0
  for i, (inputs, labels) in enumerate(train_loader):
    inputs = inputs.to(device)
    labels = labels.to(device)

    optimizer.zero_grad()

    outputs = model(inputs)
    loss = criterion(outputs, labels)

    loss.backward()
    optimizer.step()
    running_loss += loss.item() * inputs.size(0)
  epoch_loss = running_loss / len(train_dataset)
  print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {epoch_loss:.4f}")

Epoch 1/10, Loss: 97.5278
Epoch 2/10, Loss: 2.1114
Epoch 3/10, Loss: 0.7060
Epoch 4/10, Loss: 0.0670
Epoch 5/10, Loss: 0.0225
Epoch 6/10, Loss: 0.0217
Epoch 7/10, Loss: 0.0046
Epoch 8/10, Loss: 0.0105
Epoch 9/10, Loss: 0.0234
Epoch 10/10, Loss: 0.0153


In [19]:
# Evaluate
model.eval()
X_test = X_test.to(device)
Y_test = Y_test.to(device)

with torch.no_grad():
  outputs = model(X_test)
  _, predicted = torch.max(outputs.data, 1)
  accuracy = (predicted == Y_test).float().mean()

print("Test Accuracy:", accuracy.item())

Test Accuracy: 0.42499998211860657
